##  结构化输出： schema 参数

In [5]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from rich import print as rprint

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)
# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}}  # 关键：永久关闭思考模式

)



### 输出模式 1 ： Pydantic 类型
Pydantic 类型的 Schema 支持数据验证， 是优先推荐使用的方式 。

In [6]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
      """用户的联系方式"""

      name: str = Field(description=" 用户姓名 ")
      email: str = Field(description=" 用户邮箱地址 ")
      phone: str = Field(description=" 用户的手机号 ")


agent = create_agent(model=model, response_format=ToolStrategy(ContactInfo))

response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912"
                  )
            ]
      }
)

for msg in response["messages"]:
      msg.pretty_print()

      # print(response["structured_response"])

================================ Human Message =================================

 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_zYOE8jy7UWpUqc7WErx74517)
 Call ID: call_00_zYOE8jy7UWpUqc7WErx74517
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'


In [ ]:
### 举例 2：

In [10]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
      """
      在客户数据库中搜索信息

      Args:
          query (str): 客户查询字符串，例如 "张三" 或 "李四"

      Returns:
          str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
      """
      # 模拟数据库查询结果
      if "张三" in query.lower():
            return "客户记录：张三，VIP 客户，最近购买日期：2026-01-15，累计消费：$15,000"
      elif "李四" in query.lower():
            return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
      else:
            return f"关于客户 {query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
      """
      发送感谢邮件

      Args:
          customer (str): 客户名称，例如 "张三" 或 "李四"

      Returns:
          str: 确认消息，包含已发送的客户名称
      """
      return f"已向 {customer} 发送感谢邮件"


# 定义 Pydantic Schema
class CustomerAnalysis(BaseModel):
      """客户分析报告"""

      customer_name: str = Field(None, description="客户姓名")
      customer_tier: Literal[
            "潜在客户", "普通客户", "VIP 客户", "流失风险"
      ] = Field(
            "潜在客户",
            description="客户等级，只能是潜在客户、普通客户、VIP 客户或流失风险",
      )
      recent_activity: str = Field(None, description="最近活动")
      spending_level: Literal["低", "中", "高"] = Field(
            None, description="消费水平"
      )
      send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
      model=model,
      system_prompt=SystemMessage(
            content=(
                  "请分析指定客户的情况："
                  "1. 先搜索客户数据库了解最新情况 "
                  "2. 如果是 VIP 客户，则发送感谢邮件 "
                  "3. 基于搜索结果生成结构化分析报告 "
                  "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
            )
      ),
      tools=[search_customer_database, send_email],
      response_format=ToolStrategy(CustomerAnalysis),
)

# 执行分析
result = agent.invoke(
      {
            "messages": [{"role": "user", "content": "请分析客户张三"}]
            # "messages": [{"role": "user", "content": "请分析客户李四"}]
            # "messages": [{"role": "user", "content": "请分析客户王五"}]
            # "messages": [{"role": "user", "content": "今天天气如何"}]
      }
)

# 处理结果
# rprint(result)

if "structured_response" in result:
      analysis = result["structured_response"]
      print(analysis)

customer_name='张三' customer_tier='VIP 客户' recent_activity='最近购买日期：2026-01-15' spending_level='高' send_email=True


### 输出模式 2 ： TypedDict 类型

In [11]:
from typing import TypedDict, Annotated
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage


class ContactInfo(TypedDict):
      """用户的联系方式"""

      name: Annotated[str, ..., " 用户姓名 "]
      email: Annotated[str, ..., " 用户邮箱地址 "]
      phone: Annotated[str, ..., " 用户的手机号 "]


agent = create_agent(model=model, response_format=ToolStrategy(ContactInfo))

response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912"
                  )
            ]
      }
)

for msg in response["messages"]:
      msg.pretty_print()

      # print(response["structured_response"])

================================ Human Message =================================

 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_0dVMrnP0zQkf4WLjd7Uo1127)
 Call ID: call_00_0dVMrnP0zQkf4WLjd7Uo1127
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'}


### 举例 2：

In [14]:
from langchain_core.messages import SystemMessage
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
      """
      在客户数据库中搜索信息

      Args:
      query (str): 客户查询字符串，例如 " 张三 " 或 " 李四 "

      Returns:
      str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
      """
      # 模拟数据库查询结果
      if "张三" in query.lower():
            return (
                  " 客户记录：张三， VIP 客户，最近购买日期： 2026-01-15 ，累计消费：$15,000"
            )
      elif "李四" in query.lower():
            return (
                  " 客户记录：李四，普通客户，最近购买日期： 2025-12-20 ，累计消费： $3,200"
            )
      else:
            return f" 关于客户 {query} ，无记录 "


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
      """
      发送感谢邮件

      Args:
      customer (str): 客户名称，例如 " 张三 " 或 " 李四 "

      Returns:
      str: 确认消息，包含已发送的客户名称
      """
      return f" 已向 {customer} 发送感谢邮件 "

      # 使用 TypedDict 定义客户分析报告 Schema
      class CustomerAnalysis(TypedDict):
            """客户分析报告"""

            customer_name: Annotated[Optional[str], None, " 客户姓名 "]
            customer_tier: Annotated[
                  Literal["潜在客户", "普通客户", "VIP 客户", "流失风险"],
                  "潜在客户",
                  "客户等级",
            ]
            recent_activity: Annotated[Optional[str], None, "最近活动"]


spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消费水平"]
send_email: Annotated[bool, False, "是否已发送感谢邮件"]

# 创建智能体
agent = create_agent(
      model=model,
      system_prompt=SystemMessage(
            content=""
                    " 请分析指定客户的情况： "
                    "1. 先搜索客户数据库了解最新情况 "
                    "2. 如果是 VIP 客户，则发送感谢邮件 "
                    "3. 基于搜索结果生成结构化分析报告 "
                    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件 "
      ),
      tools=[search_customer_database, send_email],
      response_format=ToolStrategy(CustomerAnalysis),
)

# 执行分析
result = agent.invoke(
      {
            "messages": [{"role": "user", "content": " 请分析客户张三 "}]
            # "messages": [{"role": "user","content": " 请分析客户李四 "}]
            # "messages": [{"role": "user","content": " 请分析客户王五 "}]
            # "messages": [{"role": "user","content": " 今天天气如何 "}]
      }
)

# 处理结果
# print("result:", result)
if "structured_response" in result:
      analysis = result["structured_response"]
      print(analysis)

customer_name='张三' customer_tier='VIP 客户' recent_activity='最近购买日期：2026-01-15' spending_level='高' send_email=True


### 输出模式 3 ： JsonSchema 类型
JSON Schema 是提供一个标准的 JSON Schema 字典来定义结构。适合需要与多种编程语言交互或进行
复杂数据约束定义的场景。

In [19]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

json_schema = {
      "title": "ContactInfo",
      "description": " 用户的联系方式 ",
      "type": "object",
      "properties": {
            "name": {"description": " 用户姓名 ", "type": "string"},
            "email": {"description": " 用户邮箱地址 ", "type": "string"},
            "phone": {"description": " 用户的手机号 ", "type": "string"},
      },
      "required": ["name", "email", "phone"],
}

agent = create_agent(model=model, response_format=ToolStrategy(json_schema))

response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912"
                  )
            ]
      }
)

for msg in response["messages"]:
      msg.pretty_print()

      # print(response["structured_response"])

================================ Human Message =================================

 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_11Jb3KImhM3oUX1yXCjw0560)
 Call ID: call_00_11Jb3KImhM3oUX1yXCjw0560
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'}


**举例 2：**

In [21]:
from langchain_core.messages import SystemMessage
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
      """
      在客户数据库中搜索信息

      Args:
      query (str): 客户查询字符串，例如 " 张三 " 或 " 李四 "

      Returns:
      str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
      """
      # 模拟数据库查询结果
      if "张三" in query.lower():
            return (
                  " 客户记录：张三， VIP 客户，最近购买日期： 2026-01-15 ，累计消费：$15,000"
            )
      elif "李四" in query.lower():
            return (
                  " 客户记录：李四，普通客户，最近购买日期： 2025-12-20 ，累计消费： $3,200"
            )
      else:
            return f" 关于客户 {query} ，无记录 "


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
      """
      发送感谢邮件

      Args:
      customer (str): 客户名称，例如 " 张三 " 或 " 李四 "

      Returns:
      str: 确认消息，包含已发送的客户名称
      """
      return f" 已向 {customer} 发送感谢邮件 "


# 定义 JSON Schema 替代 Pydantic 模型
customer_analysis_schema = {
      "title": "CustomerAnalysis",
      "type": "object",
      "description": " 客户分析报告 ",
      "properties": {
            "customer_name": {"type": "string", "default": "", "description": " 客户姓名 "},
            "customer_tier": {
                  "type": "string",
                  "enum": [" 潜在客户 ", " 普通客户 ", "VIP 客户 ", " 流失风险 "],
                  "default": " 潜在客户 ",
                  "description": " 客户等级 ",
            },
            "recent_activity": {
                  "type": "string",
                  "default": "",
                  "description": " 最近活动 ",
            },
            "spending_level": {
                  "type": "string",
                  "enum": [" 低 ", " 中 ", " 高 "],
                  "default": " 低 ",
                  "description": " 消费水平 ",
            },
            "send_email": {
                  "type": "boolean",
                  "default": False,
                  "description": " 是否已发送感谢邮件 ",
            },
      },
      # 所有字段都是必须输出的
      "required": ["customer_name", "customer_tier", "recent_activity", "spending_level"],
}

# 创建智能体
agent = create_agent(
      model=model,
      system_prompt=SystemMessage(
            content=""
                    " 请分析指定客户的情况： "
                    "1. 先搜索客户数据库了解最新情况 "
                    "2. 如果是 VIP 客户，则发送感谢邮件 "
                    "3. 基于搜索结果生成结构化分析报告 "
                    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件 "
      ),
      tools=[search_customer_database, send_email],
      response_format=ToolStrategy(customer_analysis_schema),
)

# 执行分析
result = agent.invoke(
      {
            "messages": [{"role": "user", "content": " 请分析客户张三 "}]
            # "messages": [{"role": "user","content": " 请分析客户李四 "}]
            # "messages": [{"role": "user","content": " 请分析客户王五 "}]
            # "messages": [{"role": "user","content": " 今天天气如何 "}]
      }
)

# 处理结果
# print("result:", result)
if "structured_response" in result:
      analysis = result["structured_response"]
      rprint(analysis)

{
    'customer_name': '张三',
    'customer_tier': 'VIP 客户',
    'recent_activity': '最近购买日期：2026-01-15',
    'spending_level': '高',
    'send_email': True
}

### 输出模式 4 ： @dataclass 类型

In [22]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.messages import HumanMessage


@dataclass
class ContactInfo:
      """用户的联系方式"""

      name: str = Field(description=" 用户姓名 ")
      email: str = Field(description=" 用户邮箱地址 ")
      phone: str = Field(description=" 用户的手机号 ")


agent = create_agent(model=model, response_format=ToolStrategy(ContactInfo))

response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912"
                  )
            ]
      }
)

for msg in response["messages"]:
      msg.pretty_print()

      # print(response["structured_response"])

================================ Human Message =================================

 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_nk46LHUkvmA6WsfnGOvX1247)
 Call ID: call_00_nk46LHUkvmA6WsfnGOvX1247
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')


**举例 2：**

In [24]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
      """
      在客户数据库中搜索信息

      Args:
      query (str): 客户查询字符串，例如 " 张三 " 或 " 李四 "

      Returns:
      str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
      """
      # 模拟数据库查询结果
      if "张三" in query.lower():
            return (
                  " 客户记录：张三， VIP 客户，最近购买日期： 2026-01-15 ，累计消费：$15,000"
            )
      elif "李四" in query.lower():
            return (
                  " 客户记录：李四，普通客户，最近购买日期： 2025-12-20 ，累计消费： $3,200"
            )
      else:
            return f" 关于客户 {query} ，无记录 "


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
      """
      发送感谢邮件

      Args:
      customer (str): 客户名称，例如 " 张三 " 或 " 李四 "

      Returns:
      str: 确认消息，包含已发送的客户名称
      """
      return f" 已向 {customer} 发送感谢邮件 "


# 使用 Dataclass 定义 Schema
@dataclass
class CustomerAnalysis:
      """客户分析报告"""

      customer_name: str = Field(None, description=" 客户姓名 ")
      customer_tier: Literal[" 潜在客户 ", " 普通客户 ", "VIP 客户 ", " 流失风险 "] = (
            Field(
                  " 潜在客户 ",
                  description=" 客户等级 , 只能是潜在客户、普通客户、 VIP 客户或流失风险 ",
            )
      )
      recent_activity: str = Field(None, description=" 最近活动 ")
      spending_level: Literal[" 低 ", " 中 ", " 高 "] = Field(
            None, description=" 消费水平 "
      )
      send_email: bool = Field(False, description=" 是否已发送感谢邮件 ")


# 创建智能体
agent = create_agent(
      model=model,
      system_prompt=SystemMessage(
            content=""
                    " 请分析指定客户的情况： "
                    "1. 先搜索客户数据库了解最新情况 "
                    "2. 如果是 VIP 客户，则发送感谢邮件 "
                    "3. 基于搜索结果生成结构化分析报告 "
                    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件 "
      ),
      tools=[search_customer_database, send_email],
      response_format=ToolStrategy(CustomerAnalysis),
)

# 执行分析
result = agent.invoke(
      {
            "messages": [{"role": "user", "content": " 请分析客户张三 "}]
            # "messages": [{"role": "user","content": " 请分析客户李四 "}]
            # "messages": [{"role": "user","content": " 请分析客户王五 "}]
            # "messages": [{"role": "user","content": " 今天天气如何 "}]
      }
)

# 处理结果
# print("result:", result)
if "structured_response" in result:
      analysis = result["structured_response"]
      print(analysis)

CustomerAnalysis(customer_name='张三', customer_tier='VIP 客户 ', recent_activity='最近购买日期：2026-01-15', spending_level=' 高 ', send_email=True)


### 多 schema 联合模式
ToolStrategy 允许指定多个类型 “ Union[ 类型 1, 类型 2] ” 这种写法， LLM 能够根据输入文本的内容，智能
地选择 最合适的一个 数据模型（ Schema ）来生成结构化输出，但是最终会只有一种类型输出。

In [26]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
      """用户的联系方式"""

      name: str = Field(description=" 用户姓名 ")
      email: str = Field(description=" 用户邮箱地址 ")
      phone: str = Field(description=" 用户的手机号 ")


class EventInfo(BaseModel):
      """事件详情"""

      event_name: str = Field(description=" 事件名称 ")
      date: str = Field(description=" 事件发生日期 ")


agent = create_agent(
      model=model, response_format=ToolStrategy(Union[ContactInfo, EventInfo])
)

response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息： 2026 年高考报名人数突破 1200 万"
                  )
            ]
      }
)

for msg in response["messages"]:
      msg.pretty_print()

print(response["structured_response"])

================================ Human Message =================================

 从这段话中抽取结构化信息： 2026 年高考报名人数突破 1200 万
================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_00_01BXqEavHURrZi8yZGDq4241)
 Call ID: call_00_01BXqEavHURrZi8yZGDq4241
  Args:
    event_name: 高考报名人数突破1200万
    date: 2026 年
================================= Tool Message =================================
Name: EventInfo

Returning structured response: event_name='高考报名人数突破1200万' date='2026 年'
event_name='高考报名人数突破1200万' date='2026 年'


### 自定义工具消息： tool_message_content 参数
angChain 会在消息列表末尾追加一条Tool_message ，让整个链路完整

In [29]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint


class ContactInfo(BaseModel):
      """用户的联系方式"""

      name: str = Field(description=" 用户姓名 ")
      email: str = Field(description=" 用户邮箱地址 ")
      phone: str = Field(description=" 用户的手机号 ")


agent = create_agent(model=model, response_format=ToolStrategy(ContactInfo))

response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912"
                  )
            ]
      }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content=' 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912',
            additional_kwargs={},
            response_metadata={},
            id='ada15c19-d32f-49df-8dad-f5fca1a98d7a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 73,
                    'prompt_tokens': 345,
                    'total_tokens': 418,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 217
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': 'b5485277-1447-4a2a-9c0d-48267ebb811a',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d619-6224-76d9-bb33-c6723aab4ab3-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_srnt57bpRSDplqiKpAKn5041',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 345,
                'output_tokens': 73,
                'total_tokens': 418,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='7bc12d7c-2106-480b-879e-ca9184ddab4a',
            tool_call_id='call_00_srnt57bpRSDplqiKpAKn5041'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

### 自定义 tool_message_content

In [28]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint


class ContactInfo(BaseModel):
      """用户的联系方式"""

      name: str = Field(description=" 用户姓名 ")
      email: str = Field(description=" 用户邮箱地址 ")
      phone: str = Field(description=" 用户的手机号 ")


agent = create_agent(
      model=model,
      response_format=ToolStrategy(ContactInfo, tool_message_content=" 已成功抽取信息 "),
)

response = agent.invoke(
      {
            "messages": [
                  HumanMessage(
                        " 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912"
                  )
            ]
      }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content=' 从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com ，手机号： 12345678912',
            additional_kwargs={},
            response_metadata={},
            id='dcddbc6a-e7f2-442b-b785-7160af9e7ce3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 73,
                    'prompt_tokens': 345,
                    'total_tokens': 418,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 217
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': 'f02e3917-adcd-419c-a3df-570ce7b0997d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d618-7975-7743-883c-b2b2faee2ef0-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_Qc2eqshlCZg0ysedqzYK9101',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 345,
                'output_tokens': 73,
                'total_tokens': 418,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content=' 已成功抽取信息 ',
            name='ContactInfo',
            id='5f3f7abf-ede8-417e-8ef0-980286906885',
            tool_call_id='call_00_Qc2eqshlCZg0ysedqzYK9101'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

## 错误处理
###  handle_errors 参数    **情况 1：** 设置为 True/False/ 固定字符串

In [38]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint


class ContactInfo(BaseModel):
      """个人联系信息"""

      name: str = Field(description=" 姓名 ")
      email: str = Field(description=" 电子邮箱 ")


class EventDetails(BaseModel):
      """活动详情"""

      event_name: str = Field(description=" 活动名称 ")
      date: str = Field(description=" 活动日期 ")


agent = create_agent(
      model=model,
      response_format=ToolStrategy(
            Union[ContactInfo, EventDetails],
            tool_message_content=" 提取完成！ ",
            handle_errors=True,  # 当大模型生成的格式不符合要求时，它把错误当成提示词丢回给大模型，让大模型“知错能改”，而不是直接让程序崩溃。
            # handle_errors=" 请检查输入数据 "
      ),
)

result = agent.invoke(
      {
            "messages": [
                  {
                        "role": "user",
                        "content": f" 请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com ，活动名称：公司年会，活动日期： 2026-07-15",
                  }
            ]
      }
)

rprint(result)

# for msg in result["messages"]:
#       msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

### **情况 3：** 设置为自定义错误处理函数

In [39]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent

from langchain.agents.structured_output import (
    ToolStrategy,
    StructuredOutputValidationError,
    MultipleStructuredOutputsError,
)
from rich import print as rprint


# 自定义错误处理函数
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理器"""
    error_str = str(error)

    print(f" 捕获到错误类型： {type(error).__name__}")
    print(f" 错误详情： {error_str}")

    if isinstance(error, StructuredOutputValidationError):
        return " 数据格式有误，请检查字段是否符合要求。 "
    elif isinstance(error, MultipleStructuredOutputsError):
        return " 检测到多个响应，请选择最相关的一个进行返回。 "
    else:
        return f"Error: {error_str}"


class ContactInfo(BaseModel):
    """个人联系信息"""

    name: str = Field(description=" 姓名 ")
    email: str = Field(description=" 电子邮箱 ")


class EventDetails(BaseModel):
    """活动详情"""

    event_name: str = Field(description=" 活动名称 ")
    date: str = Field(description=" 活动日期 ")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content=" 提取完成！ ",
        handle_errors=custom_error_handler,
    ),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f" 请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com ，活动名称：公司年会，活动日期： 2026-07-15",
            }
        ]
    }
)

rprint(result)

 捕获到错误类型： MultipleStructuredOutputsError
 错误详情： Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.


{
    'messages': [
        HumanMessage(
            content=' 请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com 
，活动名称：公司年会，活动日期： 2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='c6633bf6-cbb1-4d00-ba5b-f3893434200a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 106,
                    'prompt_tokens': 406,
                    'total_tokens': 512,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 150
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': '6457ae62-4969-4f6d-9808-061e66f78cea',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d627-f4fc-75e8-9043-75174c9fd564-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_00_xV85i6bixnMtUsae35OW5326',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_01_bbrXMmzfs1PZzWO5DDwM2091',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 406,
                'output_tokens': 106,
                'total_tokens': 512,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content=' 检测到多个响应，请选择最相关的一个进行返回。 ',
            name='ContactInfo',
            id='21cafb92-261a-49ce-97a7-fc12d113c097',
            tool_call_id='call_00_xV85i6bixnMtUsae35OW5326'
        ),
        ToolMessage(
            content=' 检测到多个响应，请选择最相关的一个进行返回。 ',
            name='EventDetails',
            id='e70288b6-c863-4db6-b1c4-9d967a96e9fe',
            tool_call_id='call_01_bbrXMmzfs1PZzWO5DDwM2091'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 55,
                    'prompt_tokens': 565,
                    'total_tokens': 620,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 384,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 181
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': '2eb0b857-f08b-4957-9df0-47eeb0e201ca',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d627-f985-7500-8912-8250048381d3-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                  